# Суу — fine-tune Qwen2.5-1.5B-Instruct (LoRA)

Suprep-ийн SAT Math багш "Суу"-г жижиг моделоор fine-tune хийнэ.

**Colab дээр:** Runtime → Change runtime type → **T4 GPU** → Runtime → **Run all**. Нийт ~40–50 минут (сургалт ~20 мин, үнэлгээ base + fine-tuned ~20 мин).

Алхмууд: 1) суулгах · 2) өгөгдөл татах · 3) **base** моделийг үнэлэх · 4) LoRA сургалт · 5) **fine-tuned** моделийг үнэлэх · 6) GGUF гаргаж Ollama-д ашиглах.

### Хоёр горим (доорх `MODE`)
- **`"full"`** — анхны сургалт: бүх өгөгдөл, base моделиос эхэлнэ (~40–50 мин).
- **`"update"`** — хэрэглэгчийн санал (хүн баталсан `feedback.jsonl`)-аар **өмнөх Суу-г үргэлжлүүлж сургана**: шинэ санал + хуучин өгөгдлийн 300 жишээ (мартахгүйн тулд), ~5–10 мин. Өмнөх ажиллуулалтаас татсан `suu_lora.zip`-ийг upload хийнэ.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Data comes from the repo (ai/data/tuning/*.jsonl, built by `bun ai/train/export.ts`).
REPO = "pinecone-studio/pinequest-s5-episode-2-team-12"
BRANCH = "ml/colab-ready"   # branch that has ai/data/tuning
MODE = "full"     # "full" = first training · "update" = continue previous Suu with approved user feedback
REPLAY = 300      # update mode: old examples mixed in so the model doesn't forget
import os
# ---- Google Drive backup (free Colab can disconnect; results are copied to Drive at the end) ----
SAVE_TO_DRIVE = True       # False = only download the zips
DRIVE_DIR = "/content/drive/MyDrive/suu_training"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print("Drive ok →", DRIVE_DIR)
    except Exception as e:
        SAVE_TO_DRIVE = False
        print("Drive not mounted, will only download:", e)
BASE = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/ai/data/tuning"
for f in ["train.jsonl", "val.jsonl", "eval.jsonl", "feedback.jsonl"]:
    !wget -q -O {f} {BASE}/{f}
!wc -l train.jsonl val.jsonl eval.jsonl feedback.jsonl

In [ ]:
import json
def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]
train_rows, val_rows, eval_rows = read_jsonl("train.jsonl"), read_jsonl("val.jsonl"), read_jsonl("eval.jsonl")
feedback_rows = read_jsonl("feedback.jsonl")
print(len(feedback_rows), "approved feedback examples")
if MODE == "update" and not feedback_rows:
    raise SystemExit("Батлагдсан санал алга. `bun ai/pipeline/feedback.ts export` хийгээд push хийнэ үү.")
SYSTEM_PROMPT = train_rows[0]["messages"][0]["content"]
print(len(train_rows), "train ·", len(val_rows), "val ·", len(eval_rows), "eval")
print(SYSTEM_PROMPT)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

MAX_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

if MODE == "update":
    # Continue from the previous Suu: same LoRA shape, load its weights into the fresh adapter (keeps it trainable)
    from google.colab import files
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    print("Өмнөх ажиллуулалтын suu_lora.zip-ийг сонгоно уу")
    files.upload()
    !unzip -q -o suu_lora.zip
    set_peft_model_state_dict(model, load_file("suu-lora/adapter_model.safetensors"))
    print("Өмнөх Суу ачааллаа")

## Үнэлгээ (base vs fine-tuned)
- **Хариу зөв %** — "explain" асуултад зөв эцсийн хариу гарсан эсэх
- **Hint leak %** — "hint" асуултад хариугаа хэлчихсэн эсэх (бага байх тусам сайн)
- **False praise %** — сурагч БУРУУ хариу хэлэхэд "зөв" гэж хэлсэн эсэх (бага байх тусам сайн)
- **Монгол %** — гаралтын үсгийн хэдэн хувь нь кирилл вэ

In [ ]:
import re

def generate(prompt, max_new_tokens=256):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def norm(s):
    return re.sub(r"[\s$\\{}]", "", s.lower()).replace("frac", "")

def has_answer(text, accepted, letters=True):
    t = norm(text)
    for a in accepted:
        a = norm(a)
        if not a or (not letters and re.fullmatch(r"[a-e]\)", a)):
            continue
        if re.fullmatch(r"[\d.,/-]+", a):   # plain number: don't match 2 inside 12
            if re.search(r"(?<![\d.])" + re.escape(a) + r"(?![\d])", t): return True
        elif a in t:
            return True
    return False

def mongolian_share(text):
    letters = [c for c in text if c.isalpha()]
    if not letters: return 0.0
    return sum(1 for c in letters if "\u0400" <= c <= "\u04ff") / len(letters)

PRAISE = re.compile(r"(^|[^а-яөүё])(зөв|тийм)(?![а-яөүё])|\b(correct|yes|right)\b", re.I)
NOT_RIGHT = re.compile(r"буруу|incorrect|wrong|not correct|биш|алдаа", re.I)

def false_praise(text):
    t = text.lower()[:200]
    return bool(PRAISE.search(t)) and not NOT_RIGHT.search(t)

def score(rows):
    exp = [r for r in rows if r["mode"] == "explain"]
    hint = [r for r in rows if r["mode"] == "hint"]
    wrong = [r for r in rows if r["mode"] == "wrong"]
    pct = lambda xs, f: round(100 * sum(f(x) for x in xs) / max(len(xs), 1))
    return {
        "answer_correct_%": pct(exp, lambda r: has_answer(r["output"], r["accepted"])),
        "hint_leak_%": pct(hint, lambda r: has_answer(r["output"], r["accepted"], letters=False)),
        "false_praise_%": pct(wrong, lambda r: false_praise(r["output"])),
        "mongolian_%": pct(rows, lambda r: mongolian_share(r["output"])),
    }

def evaluate(label):
    FastLanguageModel.for_inference(model)
    rows = []
    for i, item in enumerate(eval_rows):
        rows.append({**item, "output": generate(item["prompt"])})
        if i % 10 == 0: print(f"{label}: {i}/{len(eval_rows)}")
    res = {"model": label, "samples": rows,
           "all": score(rows),
           "original": score([r for r in rows if r["source"] == "original"]),
           "generated": score([r for r in rows if r["source"] == "generated"])}
    print(label, res["all"])
    return res

base = evaluate("Qwen2.5-1.5B-Instruct (base)" if MODE == "full" else "Suu (before update)")
print("\n--- base sample ---\n", base["samples"][0]["output"][:500])

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def to_text(rows):
    return Dataset.from_list([
        {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in rows
    ])

import random
if MODE == "update":
    random.seed(42)
    replay = random.sample(train_rows, min(REPLAY, len(train_rows)))
    fit_rows = feedback_rows * 2 + replay   # feedback twice: it's small and new
    EPOCHS = 2
else:
    fit_rows = train_rows + feedback_rows
    EPOCHS = 2
random.shuffle(fit_rows)
print(MODE, "training on", len(fit_rows), "examples")

FastLanguageModel.for_training(model)
import inspect
params = inspect.signature(SFTConfig.__init__).parameters
args = dict(dataset_text_field="text", per_device_train_batch_size=4, gradient_accumulation_steps=2,
            num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=5, lr_scheduler_type="linear",
            logging_steps=20, optim="adamw_8bit", weight_decay=0.01, seed=42,
            output_dir="outputs", report_to="none")
args["max_seq_length" if "max_seq_length" in params else "max_length"] = MAX_LEN
args["eval_strategy" if "eval_strategy" in params else "evaluation_strategy"] = "epoch"
common = dict(model=model, train_dataset=to_text(fit_rows), eval_dataset=to_text(val_rows), args=SFTConfig(**args))
try:
    trainer = SFTTrainer(processing_class=tokenizer, **common)
except TypeError:
    trainer = SFTTrainer(tokenizer=tokenizer, **common)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

In [ ]:
tuned = evaluate("Suu (Qwen2.5-1.5B + LoRA)" if MODE == "full" else "Suu (after update)")

METRICS = [("answer_correct_%", "Answer correct % ↑"), ("hint_leak_%", "Hint leak % ↓"),
           ("false_praise_%", "False praise % ↓"), ("mongolian_%", "Mongolian % ↑")]
lines = ["# Suu eval report", "",
         f"Train examples: {len(train_rows)} · eval prompts: {len(eval_rows)} "
         f"({len({r['questionId'] for r in eval_rows})} held-out questions, never seen in training)", "",
         "- **original** = held-out hand-written questions (hardest test: new question styles)",
         "- **generated** = held-out template questions (same kinds as training, new numbers)", ""]
for part in ("all", "original", "generated"):
    lines += [f"## {part}", "", "| Model | " + " | ".join(m[1] for m in METRICS) + " |", "|---" * (len(METRICS) + 1) + "|"]
    for r in (base, tuned):
        lines.append(f"| {r['model']} | " + " | ".join(str(r[part][m[0]]) for m in METRICS) + " |")
    lines.append("")
lines += ["", "## Same prompt, before vs after", ""]
for b, t in [p for p in zip(base["samples"], tuned["samples"]) if p[0]["mode"] != "hint"][:6]:
    lines += [f"### {b['questionId']} · {b['mode']}", "", "**Base:**", "", b["output"], "", "**Suu:**", "", t["output"], ""]
report = "\n".join(lines)
open("eval_report.md", "w", encoding="utf-8").write(report)
from IPython.display import Markdown, display
display(Markdown(report))

## Туршиж үзэх

In [ ]:
FastLanguageModel.for_inference(model)
print(generate("If $2x+3=11$, what is $x$?\n\nЗөвлөгөө өгөөч, хариуг нь битгий хэлээрэй."))

## GGUF гаргах → Ollama

Доорх cell `suu.zip` татна (~1 GB). Laptop дээрээ задлаад:

```bash
cd suu
ollama create suu -f Modelfile
ollama run suu "Сайн уу, Суу!"
```

Дараа нь апп-ийн `.env.local`-д `OLLAMA_URL=http://localhost:11434` ба `OLLAMA_MODEL=suu` гэж бичнэ.

In [ ]:
import glob, os, shutil
model.save_pretrained("suu-lora"); tokenizer.save_pretrained("suu-lora")   # small LoRA adapter (backup)
model.save_pretrained_gguf("suu-gguf", tokenizer, quantization_method="q4_k_m")

os.makedirs("suu", exist_ok=True)
gguf = sorted(glob.glob("suu-gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf"), key=os.path.getsize)[-1]
shutil.copy(gguf, "suu/suu.gguf")
sys_prompt = SYSTEM_PROMPT.replace('"', '\\"')
modelfile = f'''FROM ./suu.gguf
TEMPLATE """{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{- range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
"""
SYSTEM "{sys_prompt}"
PARAMETER temperature 0.3
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
'''
open("suu/Modelfile", "w", encoding="utf-8").write(modelfile)
shutil.copy("eval_report.md", "suu/eval_report.md")
!zip -r -q suu.zip suu
!zip -r -q suu_lora.zip suu-lora
!ls -lh suu suu.zip suu_lora.zip
# ---- Copy results to Google Drive (safe even if the downloads fail or the tab closes) ----
import shutil, os, time
if SAVE_TO_DRIVE:
    stamp = time.strftime("%Y%m%d-%H%M")
    for f in ['suu_lora.zip', 'suu.zip']:
        if os.path.exists(f):
            shutil.copy(f, f"{DRIVE_DIR}/{stamp}_{f}")
            print("saved to Drive:", f"{DRIVE_DIR}/{stamp}_{f}")

from google.colab import files
files.download("suu.zip")
files.download("suu_lora.zip")   # keep this! "update" mode starts from it next time